# t0211: Elsys tpc5 (ECR dual mode) → Labquake Explorer HDF5

Preprocessing for an experiment recorded entirely on the Elsys TranAX system.
Each run is one `.tpc5` file recorded in ECR *dual* mode:

* **block 1** is the continuous low-rate record of the whole run (2 kHz here);
* **blocks 2…N** are high-rate records (2 MHz, 0.1 s before and 0.5 s after each trigger).

All blocks share one clock (`t = trigger_time + (i - trigger_sample) / sample_rate`).
This notebook reads block 1 of every run into a run dict, keeps the table of trigger
blocks so that Labquake Explorer can later pull the high-rate data of a picked event
straight from the tpc5 file, and writes everything into one HDF5 file that the
explorer loads.

Run sheet (2026-08-28, 21 °C, 58 % RH, flow rate 8 ml/min):

| Elsys channel | sensor        | range        |
|---------------|---------------|--------------|
| A1–A4         | PZT           | ±1 V (run1), ±0.5 V (run2–5), ±0.25 V (run6) |
| A5, A6        | pressure      | 0–10 V       |
| A7            | LVDT          | −6 to 14 V   |
| B1–B8         | eddy current  | ±25 V        |

Runs: 01 8 MPa, 02 10 MPa, 03 12 MPa, 04 14 MPa, 05 16 MPa, 06 8 MPa.

In [ ]:
import os
import re
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import h5py
import matplotlib.pyplot as plt
%matplotlib widget

# repository root, in case labquake_explorer is not pip-installed
sys.path.insert(0, str(Path.cwd().resolve().parents[1]))
from labquake_explorer.utils import tpc5
from labquake_explorer.data.data_manager import DataManager

## Paths

In [ ]:
EXPERIMENT = 't0211'
EXPERIMENT_DIR = Path('/Users/hueyke/Library/CloudStorage/SynologyDrive-data/Experiment/T0211')
DATA_DIR = EXPERIMENT_DIR / f'{EXPERIMENT}.exp' / 'data'

# Output read by Labquake Explorer. tpc5 paths are stored relative to this file,
# so keep it next to the raw data (or move both together).
OUTPUT_FILE = EXPERIMENT_DIR / f'{EXPERIMENT}.h5'

RUN_FILES = sorted(DATA_DIR.glob(f'{EXPERIMENT}_*.tpc5'))
for p in RUN_FILES:
    print(f'{p.stat().st_size / 1e6:8.1f} MB  {p.name}')

## What is in a file

Channel and block tables of the first run (metadata only, nothing large is read).

In [ ]:
with h5py.File(RUN_FILES[0], 'r') as f:
    channels = pd.DataFrame(tpc5.list_channels(f))
    blocks = pd.DataFrame([b.as_dict() for b in tpc5.list_blocks(f)])
display(channels)
display(blocks)

## Channel roles and calibration

`CHANNELS` maps the Elsys channel to the field name stored in the run (volts).
`CALIBRATION` derives the physical channels Labquake Explorer looks for
(`normal_stress`, `shear_stress`, `LP_displacement`, `displacement`) as
`factor * volts + offset`.

Stress conversions follow the lab's pressure-transducer calibration
(Tseng 2026, NTU thesis, Appendix B.3.3 and Table B.1). t0211 is a 1D fault with
a 5 cm cross-section: the normal-stress transducer (A5) reads
`V = 0.222 * sigma_n + 0.091`, i.e. `sigma_n = 4.507 V - 0.410` MPa, which the
six runs reproduce to within 1 % of their nominal 8-16 MPa. The shear jack (A6)
follows `V = 0.109 * tau` for the 5 cm section (`0.217` for 10 cm, `1.087` for
the 2D 50 x 50 cm model); the thesis gives no intercept for it, so none is
applied. The eddy-current sensors follow the lab's stage calibration, `d (mm) = slope * V +
intercept`, with slopes of -94.7, -96.6 and -95.4 um/V for the sensors on B1, B2 and
B3 (R² >= 0.9997, RMSE <= 10 um, travel up to 2 mm); the other five sensors get the
mean slope until they are calibrated. The intercepts describe the calibration jig, so
each `slip_k` channel is zeroed on the first 0.5 s of the run instead, which is also
how the experiments are run (sensors zeroed after the run-in). The slope is negative:
the voltage falls as the gap opens, so slip accumulates positive. `displacement`, the
field the Event Analyzer uses, is the mean of the eight slip channels by default
(`DISPLACEMENT_SOURCE`). The LVDT slope is still unknown and stays in volts.

In [ ]:
CHANNELS = {
    'A1': 'pzt_1', 'A2': 'pzt_2', 'A3': 'pzt_3', 'A4': 'pzt_4',
    'A5': 'pressure_1', 'A6': 'pressure_2',
    'A7': 'lvdt',
    'B1': 'eddy_1', 'B2': 'eddy_2', 'B3': 'eddy_3', 'B4': 'eddy_4',
    'B5': 'eddy_5', 'B6': 'eddy_6', 'B7': 'eddy_7', 'B8': 'eddy_8',
}

# 1D fault, 5 cm cross-section (Tseng 2026, Appendix B.3.3 / Table B.1)
NORMAL_V_PER_MPA, NORMAL_OFFSET_V = 0.222, 0.091   # A5: V = 0.222 sigma_n + 0.091
SHEAR_V_PER_MPA = 0.109                            # A6: V = 0.109 tau (no intercept given)

CALIBRATION = {
    # field: source volts, factor, offset, unit of the result  (value = factor * V + offset)
    'normal_stress':   dict(source='pressure_1', factor=1 / NORMAL_V_PER_MPA,
                            offset=-NORMAL_OFFSET_V / NORMAL_V_PER_MPA, unit='MPa'),
    'shear_stress':    dict(source='pressure_2', factor=1 / SHEAR_V_PER_MPA, offset=0.0, unit='MPa'),
    'LP_displacement': dict(source='lvdt',       factor=1.0, offset=0.0, unit='V'),  # um per V: to fill in
}

# Eddy-current sensors: the lab calibration gives d = slope * V + intercept with d in
# mm (three sensors, R^2 >= 0.9997, RMSE <= 10 um, travel up to 2 mm). The intercepts
# belong to the calibration jig, so every slip channel is zeroed on the first
# ZERO_WINDOW_S of the run instead; sensors without their own line use the mean slope.
EDDY_MM_PER_V = {'eddy_1': -0.094727676, 'eddy_2': -0.096627292, 'eddy_3': -0.095384964}   # B1, B2, B3
EDDY_MM_PER_V_DEFAULT = float(np.mean(list(EDDY_MM_PER_V.values())))
ZERO_WINDOW_S = 0.5             # slip channels are zeroed on the mean of this first interval
DISPLACEMENT_SOURCE = 'mean'    # 'mean' of all slip channels, or one of them, e.g. 'slip_5'


## Read block 1 of every run

Each run dict holds the time axis (seconds from the start of that file), every
channel in volts, the calibrated fields, and a `strain` record that says where
the high-rate data lives: the tpc5 file (relative to the output file), the
continuous block that was read, and the trigger-block table (block number,
sample rate, trigger time, start and end on the same clock).

In [ ]:
def add_slip_channels(run: dict, units: dict) -> None:
    """slip_k (um, slip positive, zeroed at the run start) from each eddy_k voltage,
    and displacement from them according to DISPLACEMENT_SOURCE."""
    dt = float(np.median(np.diff(run['time'][:2000])))
    n0 = max(1, int(round(ZERO_WINDOW_S / dt)))
    slips = []
    for k in range(1, 9):
        src = f'eddy_{k}'
        if src not in run:
            continue
        v = run[src].astype(np.float64)
        slope_um_per_v = 1000.0 * EDDY_MM_PER_V.get(src, EDDY_MM_PER_V_DEFAULT)
        s = slope_um_per_v * (v - np.nanmean(v[:n0]))
        run[f'slip_{k}'] = s.astype(np.float32)
        units[f'slip_{k}'] = 'um'
        slips.append(s)
    if slips:
        d = np.mean(slips, axis=0) if DISPLACEMENT_SOURCE == 'mean' else run[DISPLACEMENT_SOURCE]
        run['displacement'] = np.asarray(d, dtype=np.float32)
        units['displacement'] = 'um'
    run['calibration'] = {
        'normal_stress': f'sigma_n (MPa) = V / {NORMAL_V_PER_MPA} - {NORMAL_OFFSET_V / NORMAL_V_PER_MPA:.3f}',
        'shear_stress': f'tau (MPa) = V / {SHEAR_V_PER_MPA}',
        'eddy_mm_per_v': {f'eddy_{k}': EDDY_MM_PER_V.get(f'eddy_{k}', EDDY_MM_PER_V_DEFAULT) for k in range(1, 9)},
        'slip_zero_window_s': ZERO_WINDOW_S,
        'displacement_source': DISPLACEMENT_SOURCE,
    }


RUN_NAME = re.compile(rf'{EXPERIMENT}_(?P<index>\d+)_(?P<sigma>[\d.]+)MPa_(?P<run>run\d+)')


def load_run(path: Path) -> dict:
    rec = tpc5.read_continuous(path)            # block 1 only, float32 volts
    m = RUN_NAME.match(path.stem)
    run = {
        'name': m['run'] if m else path.stem,
        'file': path.name,
        'start_time': rec['start_time'],
        'normal_stress_level': float(m['sigma']) if m else float('nan'),
        'time': rec['time'].astype(np.float64),
    }
    units = {}
    for info, volts in zip(rec['channels'], rec['data']):
        short = info['name'].split()[0]          # 'A1 (0.A1)' -> 'A1'
        field = CHANNELS.get(short, short)
        run[field] = volts
        units[field] = 'V'
    for field, cal in CALIBRATION.items():
        run[field] = (cal['factor'] * run[cal['source']].astype(np.float64) + cal['offset']).astype(np.float32)
        units[field] = cal['unit']
    if units['shear_stress'] == units['normal_stress'] != 'V':
        with np.errstate(divide='ignore', invalid='ignore'):
            run['friction'] = run['shear_stress'] / run['normal_stress']
        units['friction'] = ''
    add_slip_channels(run, units)
    run['units'] = units
    run['strain'] = {
        'format': 'tpc5',
        'filename': os.path.relpath(path, OUTPUT_FILE.parent),
        'time_offset': 0.0,                      # file clock == run time
        'channel_numbers': [c['number'] for c in rec['channels']],
        'channel_names': [c['name'] for c in rec['channels']],
        'fields': [CHANNELS.get(c['name'].split()[0], c['name']) for c in rec['channels']],
        'continuous': rec['block'].as_dict(),
        'blocks': tpc5.block_table(rec['trigger_blocks']),
    }
    return run


runs = [load_run(p) for p in RUN_FILES]
pd.DataFrame([{
    'name': r['name'], 'sigma_n (MPa)': r['normal_stress_level'], 'samples': r['time'].size,
    'duration (s)': r['time'][-1], 'trigger blocks': len(r['strain']['blocks']['block']),
    'start': r['start_time'][:19], 'displacement end (um)': float(r['displacement'][-1]),
} for r in runs])

## Overview

Pressure and LVDT channels of every run; shaded spans are the trigger blocks (where 2 MHz data exists).

In [ ]:
fig, axs = plt.subplots(len(runs), 1, figsize=(10, 2.2 * len(runs)), sharex=True)
for ax, run in zip(np.atleast_1d(axs), runs):
    ax.plot(run['time'], run['normal_stress'], lw=0.6, label=f"normal_stress ({run['units']['normal_stress']})")
    ax.plot(run['time'], run['shear_stress'], lw=0.6, label=f"shear_stress ({run['units']['shear_stress']})")
    ax2 = ax.twinx()
    ax2.plot(run['time'], run['LP_displacement'], lw=0.6, color='C2', label='LP_displacement')
    ax2.set_ylabel(f"LVDT ({run['units']['LP_displacement']})", color='C2')
    for a, b in zip(run['strain']['blocks']['start'], run['strain']['blocks']['end']):
        ax.axvspan(a, b, color='0.8', zorder=-10)
    ax.set_ylabel(f"{run['name']}  {run['normal_stress_level']:g} MPa")
    ax.legend(loc='upper left', fontsize='small')
axs[-1].set_xlabel('time (s)')
fig.tight_layout()

## Clock check

One trigger block against the continuous record of the same channel. The high-rate
burst must sit on the same time axis as the disturbance seen at 2 kHz.

In [ ]:
run_idx, block_idx, field = 0, 0, 'pzt_1'
run = runs[run_idx]
blocks = tpc5.blocks_from_table(run['strain']['blocks'])
block = blocks[block_idx]
channel = run['strain']['channel_numbers'][run['strain']['fields'].index(field)]
with h5py.File(DATA_DIR / run['file'], 'r') as f:
    t_hi, v_hi = tpc5.read_window(f, block, block.trigger_time - 0.005, block.trigger_time + 0.02, channels=[channel])
sel = (run['time'] >= t_hi[0]) & (run['time'] <= t_hi[-1])

fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(t_hi, v_hi[0], lw=0.4, label=f'block {block.block} @ {block.sample_rate:g} Hz')
ax.plot(run['time'][sel], run[field][sel], 'o-', ms=3, lw=0.8, label='continuous @ 2 kHz')
ax.axvline(block.trigger_time, color='k', ls=':', label='trigger')
ax.set_xlabel('time (s)'); ax.set_ylabel(f'{field} (V)'); ax.legend(loc='upper right')
ax.set_title(f"{run['name']}: trigger block {block.block} vs continuous record")

## Experiment record and save

In [ ]:
experiment = {
    'name': EXPERIMENT,
    'date': '2026-08-28',
    'operators': ['朱展穎'],
    'temperature_C': 21.0,
    'humidity_percent': 58.0,
    'flow_rate_ml_min': 8.0,
    'acquisition': ('Elsys TranAX ECR dual mode: block 1 continuous at 2 kHz, '
                    'trigger blocks at 2 MHz with 0.1 s pre- and 0.5 s post-trigger; '
                    'A1-A4 PZT, A5-A6 pressure, A7 LVDT, B1-B8 eddy current'),
    'runs': runs,
}

dm = DataManager()
dm.data = experiment
dm.save_file(OUTPUT_FILE)
print(f'{OUTPUT_FILE}  {OUTPUT_FILE.stat().st_size / 1e6:.1f} MB')

## Check the file the way the explorer loads it

In [ ]:
check = DataManager()
check.load_file(OUTPUT_FILE)
rows = []
for i, run in enumerate(check.get_data('runs')):
    tpc5_path = OUTPUT_FILE.parent / run['strain']['filename']
    rows.append({
        'run': run['name'], 'samples': len(run['time']), 'fields': len([k for k, v in run.items() if isinstance(v, np.ndarray)]),
        'trigger blocks': len(run['strain']['blocks']['block']), 'tpc5 found': tpc5_path.exists(),
    })
pd.DataFrame(rows)

## Next

1. Open `t0211.h5` in Labquake Explorer, right-click `runs/[i]/shear_stress` → **Pick Events**, then `event_indices` → **Extract Events**.
2. Event extraction for `strain.format == 'tpc5'` (reading the trigger block that contains each event time) is the next step on the explorer side.